# LegalQA — targeted fixes smoke (12 câu)

Notebook này chỉ kiểm tra nhanh các lỗi từ run `e24a482a461f-226bece3139d`: token-limit, refusal, extractive fallback và retrieval lệch. Notebook **không rebuild Dense**, **không chạy validation** và **không chạy full 1.000 câu**.

Yêu cầu Kaggle: bật GPU T4, thêm Dataset/Output chứa `legalqa.sqlite`, `legalqa_dense.meta.json`, vector `.faiss`/`.npy` và snapshot ba model có file `.legalqa_model.json`. Commit chứa patch phải được push lên `main` trước khi chạy.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
from collections import Counter
from pathlib import Path
from typing import Any

REPO_URL = 'https://github.com/lighth-gh/uit-dsc-2026-task2-legalqa.git'
REPO_BRANCH = 'main'
EXPECTED_COMMIT = ''  # Khuyến nghị điền SHA commit chứa patch; để trống sẽ kiểm tra marker code.
RUN_GENERATION = True  # False nếu chỉ muốn test unit + retrieval.
FORCE_RERUN = False

TARGET_GROUPS = {
    'token_limit': ['80189', '63093', '55463', '67397', '42039', '138443'],
    'refusal': ['18645', '129215', '6905'],
    'fallback': ['34235', '117399', '108017', '138443'],
}
TARGET_IDS = list(dict.fromkeys(sum(TARGET_GROUPS.values(), [])))
EXPECTED_TOP1 = {
    '80189': {'document_id': '230689', 'chunk_no': 19},
    '34235': {'document_id': '102434', 'chunk_no': 306},
    '55463': {'document_id': '58283', 'chunk_no': 10},
    '42039': {'document_id': '235672', 'chunk_no': 51},
}

EMBEDDING_MODEL_ID = 'AITeamVN/Vietnamese_Embedding_v2'
RERANKER_MODEL_ID = 'AITeamVN/Vietnamese_Reranker'
GENERATOR_MODEL_ID = 'AITeamVN/Vi-Qwen2-1.5B-RAG'
MODEL_MARKER = '.legalqa_model.json'

BM25_TOP_K = 50
DENSE_TOP_K = 50
RRF_TOP_K = 50
RERANKER_CANDIDATE_K = 20
RERANK_TOP_K = 3
RERANKER_MAX_LENGTH = 1024
MAX_INPUT_TOKENS = 7168
MAX_NEW_TOKENS = 512
TOKEN_LIMIT_RETRY_TOKENS = 768
MAX_LONG_ANSWER_WORDS = 640
SEED = 2026

KAGGLE = Path('/kaggle/working').is_dir()
INPUT_ROOT = Path('/kaggle/input') if KAGGLE else Path('.').resolve()
REPO_DIR = Path('/kaggle/working/uit-dsc-2026-task2-legalqa') if KAGGLE else Path('.').resolve()
WORK_ROOT = Path('/kaggle/working/legalqa-targeted-fixes') if KAGGLE else Path('artifacts/legalqa-targeted-fixes').resolve()
WORK_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

def read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding='utf-8'))

def write_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(path)

def run_stream(command: list[str], *, cwd: Path | None = None, log_path: Path | None = None) -> None:
    print('$', ' '.join(map(str, command)), flush=True)
    lines: list[str] = []
    process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', bufsize=1)
    assert process.stdout is not None
    for line in iter(process.stdout.readline, ''):
        print(line, end='', flush=True)
        lines.append(line)
    process.stdout.close()
    return_code = process.wait()
    if log_path is not None:
        log_path.write_text(''.join(lines), encoding='utf-8')
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

print('Target IDs:', TARGET_IDS)
print('Kaggle:', KAGGLE, '| Work:', WORK_ROOT)

In [ ]:
# 1) Checkout đúng code và chạy regression tests.
if KAGGLE:
    if (REPO_DIR / '.git').is_dir():
        run_stream(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=REPO_DIR)
    elif REPO_DIR.exists():
        raise RuntimeError(f'{REPO_DIR} tồn tại nhưng không phải Git repo; không tự xóa.')
    else:
        run_stream(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)])

COMMIT_SHA = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True, encoding='utf-8').strip()
print('Commit:', COMMIT_SHA)
if EXPECTED_COMMIT and COMMIT_SHA != EXPECTED_COMMIT:
    raise RuntimeError(f'Sai commit: expected={EXPECTED_COMMIT}, actual={COMMIT_SHA}')

pipeline_source = (REPO_DIR / 'legalqa_baseline/pipeline.py').read_text(encoding='utf-8')
text_source = (REPO_DIR / 'legalqa_baseline/text.py').read_text(encoding='utf-8')
generator_source = (REPO_DIR / 'legalqa_baseline/generator.py').read_text(encoding='utf-8')
PATCH_MARKERS = {
    'retry_360_words': 'không quá 360 từ' in pipeline_source,
    'partial_answer_audit': 'partial_answer_available' in pipeline_source and 'partial_answer' in generator_source,
    'grounded_yes_no': 'refusal_grounded_yes_no_clause' in pipeline_source,
    'step_count_route': 'focused_extractive_decisive_step_count' in pipeline_source,
    'pccc_alias': 'Trình tự báo cáo và cơ quan tiếp nhận báo cáo' in text_source,
    'fund_priority': 'sử dụng quỹ bảo hiểm tai nạn lao động, bệnh nghề nghiệp' in text_source,
}
print('Patch markers:', json.dumps(PATCH_MARKERS, ensure_ascii=False, indent=2))
if not all(PATCH_MARKERS.values()):
    raise RuntimeError('Checkout chưa chứa đầy đủ patch. Hãy commit + push code rồi chạy lại notebook.')

requirements = REPO_DIR / 'requirements-generator.txt'
if KAGGLE:
    run_stream([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], cwd=REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

unit_patterns = [
    'test_baseline.py',
    'test_storage_remaining.py',
    'test_generator.py',
    'test_long_answer_routing.py',
]
for pattern in unit_patterns:
    unit_command = [
        sys.executable, '-m', 'unittest', 'discover',
        '-s', 'tests', '-p', pattern,
    ]
    run_stream(
        unit_command,
        cwd=REPO_DIR,
        log_path=WORK_ROOT / f'{Path(pattern).stem}.log',
    )
print('UNIT GATE: PASS (4 targeted test suites / 114 tests)')

In [ ]:
# 2) Tự tìm cache trong Kaggle Input; tuyệt đối không rebuild index.
INPUT_FILES = [path for path in INPUT_ROOT.rglob('*') if path.is_file()] if INPUT_ROOT.exists() else []
print(f'Đã quét {len(INPUT_FILES):,} input files')

def artifact_rank(path: Path) -> tuple[int, int, str]:
    has_manifest = any((parent / 'legalqa_artifacts.json').is_file() for parent in path.parents)
    return (0 if has_manifest else 1, len(path.parts), str(path))

def find_named(names: set[str], *, repo_fallback: bool = False) -> Path:
    accepted = {name.casefold() for name in names}
    matches = sorted((path for path in INPUT_FILES if path.name.casefold() in accepted), key=artifact_rank)
    if matches:
        return matches[0]
    if repo_fallback:
        for name in names:
            candidate = REPO_DIR / 'data' / name
            if candidate.is_file():
                return candidate
    raise FileNotFoundError(f'Không tìm thấy artifact: {sorted(names)}')

def complete_model(folder: Path) -> bool:
    return (folder / 'config.json').is_file() and (any(folder.glob('*.safetensors')) or any(folder.glob('pytorch_model*.bin')))

def discover_model(repo_id: str) -> dict[str, Any]:
    candidates: list[dict[str, Any]] = []
    for marker in (path for path in INPUT_FILES if path.name == MODEL_MARKER):
        try:
            payload = read_json(marker)
        except Exception:
            continue
        if payload.get('repo_id') == repo_id and payload.get('revision') and complete_model(marker.parent):
            candidates.append({'repo_id': repo_id, 'revision': str(payload['revision']), 'path': marker.parent})
    cache_key = ('models--' + repo_id.replace('/', '--')).casefold()
    for config_path in (path for path in INPUT_FILES if path.name == 'config.json'):
        folder = config_path.parent
        if cache_key in [part.casefold() for part in folder.parts] and folder.parent.name == 'snapshots' and complete_model(folder):
            candidates.append({'repo_id': repo_id, 'revision': folder.name, 'path': folder})
    if not candidates:
        raise FileNotFoundError(f'Thiếu snapshot model có revision: {repo_id}')
    return sorted(candidates, key=lambda item: artifact_rank(Path(item['path'])))[0]

PUBLIC_PATH = find_named({'public-official.json', 'public_official.json', 'public_test.json'}, repo_fallback=True)
BM25_SOURCE_PATH = find_named({'legalqa.sqlite'})
DENSE_META_PATH = find_named({'legalqa_dense.meta.json'})
DENSE_INDEX_PATH = Path(str(DENSE_META_PATH).removesuffix('.meta.json'))
DENSE_VECTOR_PATH = next((path for path in (DENSE_INDEX_PATH.with_suffix('.faiss'), DENSE_INDEX_PATH.with_suffix('.npy')) if path.is_file()), None)
if DENSE_VECTOR_PATH is None:
    raise FileNotFoundError(f'Thiếu dense vector cạnh {DENSE_META_PATH}')

MODEL_INFO = {
    'embedding': discover_model(EMBEDDING_MODEL_ID),
    'reranker': discover_model(RERANKER_MODEL_ID),
    'generator': discover_model(GENERATOR_MODEL_ID),
}
HF_HUB_CACHE = WORK_ROOT / 'hf-cache'
for info in MODEL_INFO.values():
    snapshot_link = HF_HUB_CACHE / ('models--' + info['repo_id'].replace('/', '--')) / 'snapshots' / info['revision']
    snapshot_link.parent.mkdir(parents=True, exist_ok=True)
    if not snapshot_link.exists() and not snapshot_link.is_symlink():
        snapshot_link.symlink_to(Path(info['path']).resolve(), target_is_directory=True)
os.environ['HF_HUB_CACHE'] = str(HF_HUB_CACHE)
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'

LOCAL_DB_PATH = WORK_ROOT / 'legalqa.sqlite'
if not LOCAL_DB_PATH.is_file() or LOCAL_DB_PATH.stat().st_size != BM25_SOURCE_PATH.stat().st_size:
    copying = LOCAL_DB_PATH.with_suffix('.sqlite.copying')
    shutil.copyfile(BM25_SOURCE_PATH, copying)
    copying.replace(LOCAL_DB_PATH)

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Cần bật GPU Accelerator trên Kaggle.')
print('GPU:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print('Public:', PUBLIC_PATH)
print('BM25:', LOCAL_DB_PATH)
print('Dense:', DENSE_INDEX_PATH, '|', DENSE_VECTOR_PATH)
print('Models:', json.dumps({key: {**value, 'path': str(value['path'])} for key, value in MODEL_INFO.items()}, ensure_ascii=False, indent=2))

In [ ]:
# 3) Tạo input đúng 12 ID và chạy retrieval-only.
PUBLIC_DATA = read_json(PUBLIC_PATH)
if not isinstance(PUBLIC_DATA, dict):
    raise ValueError('Public data phải là object keyed by ID')
missing_ids = [qid for qid in TARGET_IDS if qid not in PUBLIC_DATA]
if missing_ids:
    raise RuntimeError(f'Thiếu target IDs: {missing_ids}')
TARGET_INPUT_PATH = WORK_ROOT / f'targeted_12_{COMMIT_SHA[:12]}.json'
write_json(TARGET_INPUT_PATH, {qid: PUBLIC_DATA[qid] for qid in TARGET_IDS})

COMMON_ARGS = [
    '--input', str(TARGET_INPUT_PATH), '--db', str(LOCAL_DB_PATH),
    '--bm25-top-k', str(BM25_TOP_K), '--dense-top-k', str(DENSE_TOP_K),
    '--rrf-k', '60', '--rrf-top-k', str(RRF_TOP_K),
    '--reranker-candidate-k', str(RERANKER_CANDIDATE_K), '--rerank-top-k', str(RERANK_TOP_K),
    '--context-top-k', str(RERANK_TOP_K), '--dense-query-max-length', '256',
    '--reranker-max-length', str(RERANKER_MAX_LENGTH),
    '--knn-threshold', '0.72', '--guarded-knn-threshold', '0.90',
    '--dense-index', str(DENSE_INDEX_PATH), '--embedding-model', EMBEDDING_MODEL_ID,
    '--embedding-revision', MODEL_INFO['embedding']['revision'],
    '--reranker-model', str(MODEL_INFO['reranker']['path']), '--device', 'auto',
]
RETRIEVAL_PATH = WORK_ROOT / f'retrieval_targeted_12_{COMMIT_SHA[:12]}.json'
RETRIEVAL_LOG = WORK_ROOT / f'retrieval_targeted_12_{COMMIT_SHA[:12]}.log'
if FORCE_RERUN or not RETRIEVAL_PATH.is_file():
    started = time.perf_counter()
    run_stream([sys.executable, '-m', 'legalqa_baseline', 'diagnose-retrieval', *COMMON_ARGS, '--output', str(RETRIEVAL_PATH)], cwd=REPO_DIR, log_path=RETRIEVAL_LOG)
    print(f'Retrieval elapsed: {time.perf_counter() - started:.1f}s')
else:
    print('Reuse:', RETRIEVAL_PATH)

retrieval_payload = read_json(RETRIEVAL_PATH)
retrieval_items = {str(item['id']): item for item in retrieval_payload.get('items', [])}
retrieval_rows = []
retrieval_expected_pass = True
for qid in TARGET_IDS:
    item = retrieval_items.get(qid, {})
    top3 = item.get('diagnostic_candidates', {}).get('top3', [])
    top = top3[0] if top3 else {}
    document_id = str(top.get('document_id') or top.get('context_id') or '')
    chunk_no = top.get('chunk_no')
    expected = EXPECTED_TOP1.get(qid)
    expected_ok = None if expected is None else (document_id == expected['document_id'] and int(chunk_no if chunk_no is not None else -1) == expected['chunk_no'])
    if expected_ok is False:
        retrieval_expected_pass = False
    retrieval_rows.append({
        'id': qid, 'status': item.get('status'), 'top1_document': document_id,
        'top1_chunk': chunk_no, 'raw_score': top.get('rerank_score'),
        'expected_top1': expected_ok, 'question': PUBLIC_DATA[qid]['question'],
    })

import pandas as pd
display(pd.DataFrame(retrieval_rows))
print('RETRIEVAL KNOWN-TARGET GATE:', 'PASS' if retrieval_expected_pass else 'FAIL')

In [ ]:
# 4) Generation smoke đúng 12 câu lỗi. Thường mất khoảng 3–8 phút trên 2xT4.
SUBMISSION_PATH = WORK_ROOT / f'submission_targeted_12_{COMMIT_SHA[:12]}.json'
AUDIT_PATH = WORK_ROOT / f'submission_targeted_12_{COMMIT_SHA[:12]}.audit.jsonl'
GENERATION_LOG = WORK_ROOT / f'generation_targeted_12_{COMMIT_SHA[:12]}.log'
CHECKPOINT_PATH = SUBMISSION_PATH.with_suffix('.checkpoint.json')

if RUN_GENERATION:
    if FORCE_RERUN or not (SUBMISSION_PATH.is_file() and AUDIT_PATH.is_file()):
        command = [
            sys.executable, '-m', 'legalqa_baseline', 'predict', *COMMON_ARGS,
            '--output', str(SUBMISSION_PATH), '--audit-output', str(AUDIT_PATH),
            '--mode', 'hybrid_rag', '--generator-model', str(MODEL_INFO['generator']['path']),
            '--max-new-tokens', str(MAX_NEW_TOKENS), '--max-input-tokens', str(MAX_INPUT_TOKENS),
            '--token-limit-retry-tokens', str(TOKEN_LIMIT_RETRY_TOKENS),
            '--max-long-answer-words', str(MAX_LONG_ANSWER_WORDS),
            '--generation-seed', str(SEED), '--checkpoint-interval', '1',
        ]
        if CHECKPOINT_PATH.is_file() and not FORCE_RERUN:
            command.append('--resume')
        started = time.perf_counter()
        run_stream(command, cwd=REPO_DIR, log_path=GENERATION_LOG)
        print(f'Generation elapsed: {time.perf_counter() - started:.1f}s')
    else:
        print('Reuse:', SUBMISSION_PATH, AUDIT_PATH)
else:
    print('SKIP generation vì RUN_GENERATION=False')

In [ ]:
# 5) Báo cáo PASS/FAIL theo từng lỗi.
if not RUN_GENERATION:
    print('RESULT: RETRIEVAL_ONLY', '| known-target gate =', retrieval_expected_pass)
else:
    from legalqa_baseline.text import (
        is_heading_only_answer, is_refusal_answer, output_artifact_flags, possibly_cut,
    )

    predictions = read_json(SUBMISSION_PATH)
    audit_rows = [json.loads(line) for line in AUDIT_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
    audit_by_id = {str(row.get('id')): row for row in audit_rows}
    result_rows = []
    for qid in TARGET_IDS:
        answer = str(predictions.get(qid, {}).get('answer') or '').strip()
        audit = audit_by_id.get(qid, {})
        flags = sorted(output_artifact_flags(answer))
        refusal = bool(is_refusal_answer(answer) or audit.get('says_no_information'))
        final_token_limit = bool(audit.get('hit_token_limit'))
        cut = bool(possibly_cut(answer) or audit.get('possibly_cut'))
        heading = bool(str(audit.get('route', '')).startswith('extractive') and is_heading_only_answer(answer))
        item_pass = bool(answer and not refusal and not final_token_limit and not cut and not heading and not flags)
        result_rows.append({
            'id': qid, 'pass': item_pass, 'route': audit.get('route'),
            'recovery': audit.get('recovery_strategy'), 'final_token_limit': final_token_limit,
            'refusal': refusal, 'possibly_cut': cut, 'artifacts': ','.join(flags),
            'words': len(answer.split()), 'seconds': round(float(audit.get('stage_seconds', {}).get('total', 0.0)), 2),
            'answer_preview': answer[:180],
        })

    token_limit_remaining = [row['id'] for row in result_rows if row['id'] in TARGET_GROUPS['token_limit'] and row['final_token_limit']]
    refusal_remaining = [row['id'] for row in result_rows if row['id'] in TARGET_GROUPS['refusal'] and row['refusal']]
    fallback_remaining = [row['id'] for row in result_rows if row['route'] == 'extractive_fallback']
    invalid_ids = [row['id'] for row in result_rows if not row['pass']]
    gates = {
        'patch_markers': all(PATCH_MARKERS.values()),
        'retrieval_known_targets_top1': retrieval_expected_pass,
        'old_token_limit_ids_fixed': not token_limit_remaining,
        'old_refusal_ids_fixed': not refusal_remaining,
        'all_12_outputs_valid': not invalid_ids,
    }
    status = 'READY_FOR_SMOKE30' if all(gates.values()) else 'NOT_FIXED_YET'
    report = {
        'status': status, 'commit_sha': COMMIT_SHA, 'gates': gates,
        'token_limit_remaining': token_limit_remaining,
        'refusal_remaining': refusal_remaining,
        'extractive_fallback_remaining': fallback_remaining,
        'invalid_ids': invalid_ids, 'routes': dict(Counter(str(row['route']) for row in result_rows)),
        'note': 'Tập 12 câu bị thiên lệch về lỗi; fallback rate ở đây không thay cho gate smoke30.',
        'items': result_rows, 'retrieval': retrieval_rows,
    }
    REPORT_PATH = WORK_ROOT / f'targeted_fixes_report_{COMMIT_SHA[:12]}.json'
    write_json(REPORT_PATH, report)
    display(pd.DataFrame(result_rows))
    print(json.dumps({key: value for key, value in report.items() if key not in {'items', 'retrieval'}}, ensure_ascii=False, indent=2))
    print('Report:', REPORT_PATH)
    print('KẾT LUẬN:', status)
    if status == 'READY_FOR_SMOKE30':
        print('Chạy tiếp smoke30; chưa chạy full 1.000 trước khi smoke30 và validation 100/300 PASS.')